# Notebook 6: Hosted Agents, Publishing & Agent-to-Agent (A2A)
## Deploy Containerized Agents, Manage Sessions, Publish to M365 & Enable A2A

**Sources:**
- [Deploy a Hosted Agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent)
- [Manage Hosted Agents (REST)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent?pivots=rest)
- [Manage Hosted Agents (Python)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent?pivots=python)
- [Manage Hosted Agents (azd)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent?pivots=azd)
- [Manage Hosted Sessions (REST)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions?pivots=rest)
- [Manage Hosted Sessions (Python)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions?pivots=python)
- [Manage Hosted Sessions (azd)](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions?pivots=azd)
- [Publish to Microsoft 365 Copilot & Teams](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/publish-copilot)
- [Microsoft Agent 365 & Foundry](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/agent-365)
- [Enable Agent-to-Agent (A2A) Endpoint](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint)

---

This notebook covers:
1. Hosted Agents — Containerized agent deployment lifecycle
2. Container requirements — Protocols, ports, environment variables
3. Deploy using Python SDK & REST API
4. Manage Hosted Agents — Versions, traffic routing, logs, identity
5. Manage Hosted Sessions — Sessions vs conversations, isolation keys, file operations
6. Publish to Microsoft 365 Copilot & Teams
7. Microsoft Agent 365 — IT admin governance & AI teammates
8. Agent-to-Agent (A2A) — Expose & connect agents via A2A protocol
9. Interview Q&A

## Master Architecture Diagram

```
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                   HOSTED AGENTS, PUBLISHING & A2A — FULL PICTURE                         │
│                                                                                          │
│  ┌──────────── DEPLOY ────────────┐  ┌──────────── MANAGE ─────────────┐                 │
│  │                                │  │                                 │                 │
│  │  1. Build Container Image      │  │  • List agents & versions       │                 │
│  │  2. Push to Azure ACR          │  │  • Poll version status          │                 │
│  │  3. Create agent version       │  │  • Configure traffic routing    │                 │
│  │  4. Poll until active          │  │  • Canary deployments (90/10)   │                 │
│  │  5. Invoke via endpoint        │  │  • View container logs          │                 │
│  │                                │  │  • Retrieve agent identity      │                 │
│  │  Protocols:                    │  │  • Delete versions/agents       │                 │
│  │  ├── Responses (/responses)    │  │                                 │                 │
│  │  └── Invocations (/invocations)│  └─────────────────────────────────┘                 │
│  └────────────────────────────────┘                                                      │
│                                                                                          │
│  ┌──────────── SESSIONS ──────────┐  ┌──────────── PUBLISH ────────────┐                 │
│  │                                │  │                                 │                 │
│  │  Session = sandbox + filesystem│  │  Microsoft 365 Copilot          │                 │
│  │  ├── 30-day persistence        │  │  ├── Direct publish             │                 │
│  │  ├── 15-min idle timeout       │  │  ├── Download & customize       │                 │
│  │  ├── VM-isolated per session   │  │  └── Admin approval flow        │                 │
│  │  └── File upload/download      │  │                                 │                 │
│  │                                │  │  Microsoft Teams                │                 │
│  │  Isolation Keys:               │  │  ├── Agent store visibility     │                 │
│  │  ├── Entra (auto-derived)      │  │  └── Sideload custom app       │                 │
│  │  └── Header (app-supplied)     │  │                                 │                 │
│  └────────────────────────────────┘  │  Agent 365 (A365)               │                 │
│                                      │  ├── IT admin governance        │                 │
│  ┌──────────── A2A ──────────────┐   │  ├── Agent registry             │                 │
│  │                               │   │  └── AI teammates               │                 │
│  │  Expose agent as A2A endpoint  │   └─────────────────────────────────┘                 │
│  │  ├── Agent card (discovery)    │                                                      │
│  │  ├── Entra ID auth required    │                                                      │
│  │  ├── HTTP+JSON / JSONRPC       │                                                      │
│  │  └── Protocol v0.3             │                                                      │
│  │                                │                                                      │
│  │  Connect FROM another agent    │                                                      │
│  │  ├── A2A connection resource   │                                                      │
│  │  └── A2APreviewTool            │                                                      │
│  └────────────────────────────────┘                                                      │
└──────────────────────────────────────────────────────────────────────────────────────────┘
```

---
## Prerequisites

| Requirement | Description | Status |
|---|---|---|
| **Azure Subscription** | Active subscription | [ ] |
| **Foundry Project** | Created in previous notebooks | [ ] |
| **Python 3.10+** | Runtime | [ ] |
| **Packages** | `azure-ai-projects>=2.1.0`, `azure-identity`, `openai` | [ ] |
| **Docker Desktop** | For building container images | [ ] |
| **Azure CLI 2.80+** | For resource management and REST calls | [ ] |
| **Azure Developer CLI 1.23+** | For `azd` workflow (optional) | [ ] |
| **Model Deployed** | gpt-4.1-mini or similar | [ ] |
| **RBAC Role** | **Foundry Project Manager** (for hosted agents) or **Foundry User** (for publishing) | [ ] |

In [ ]:
# Install required packages
!pip install "azure-ai-projects>=2.1.0" azure-identity openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "your_project_endpoint")

if PROJECT_ENDPOINT == "your_project_endpoint":
    print("WARNING: Set PROJECT_ENDPOINT in your .env file.")
    print("Format: https://<resource_name>.services.ai.azure.com/api/projects/<project_name>")
else:
    print(f"Project endpoint: {PROJECT_ENDPOINT[:60]}...")

---
# PART 1: DEPLOY A HOSTED AGENT
---

## 1.1 What Are Hosted Agents?

Hosted agents are **containerized agentic AI applications** that run on Foundry Agent Service. You package your own code into a Docker image, push it to Azure Container Registry, and the platform manages infrastructure, scaling, and identity automatically.

### Deployment Lifecycle

```
┌──────────────┐    ┌────────────────┐    ┌──────────────────┐    ┌──────────────┐
│ 1. BUILD &   │    │ 2. CREATE      │    │ 3. POLL FOR      │    │ 4. INVOKE    │
│    PUSH       │───▶│    AGENT       │───▶│    STATUS         │───▶│    AGENT     │
│              │    │    VERSION      │    │                  │    │              │
│ docker build │    │ SDK / REST     │    │ Wait for         │    │ /responses   │
│ docker push  │    │ Provisions     │    │ status: active   │    │ /invocations │
│ to ACR       │    │ infrastructure │    │                  │    │              │
└──────────────┘    └────────────────┘    └──────────────────┘    └──────────────┘
```

### Required Permissions

| Role | Purpose |
|------|--------|
| **Foundry Project Manager** | Create/deploy hosted agents + assign roles to agent identity |
| **Container Registry Repository Reader** | Project managed identity needs this to pull images |
| **Foundry User** | Platform-created agent identity needs this for runtime model/tool access |

## 1.2 Container Requirements

### Protocol Libraries

| Protocol | Python Library | .NET Library | Endpoint | Best For |
|----------|---------------|-------------|----------|----------|
| **Responses** | `azure-ai-agentserver-responses` | `Azure.AI.AgentServer.Responses` | `/responses` | Conversational chatbots, streaming, multi-turn with platform-managed history |
| **Invocations** | `azure-ai-agentserver-invocations` | `Azure.AI.AgentServer.Invocations` | `/invocations` | Webhook receivers, non-conversational processing, custom async workflows |

> A single container can expose **both protocols simultaneously**.

### Key Container Specs

| Spec | Value |
|------|-------|
| **Port** | 8088 (local); gateway handles routing in production |
| **Architecture** | `linux/amd64` required (use `--platform linux/amd64` on ARM machines) |
| **Health endpoint** | `/readiness` (auto-exposed by protocol libraries) |

### Platform-Injected Environment Variables

These are set automatically at runtime — do NOT redeclare in `agent.yaml`:

| Variable | Purpose |
|----------|--------|
| `FOUNDRY_PROJECT_ENDPOINT` | Foundry project endpoint URL |
| `FOUNDRY_PROJECT_ARM_ID` | Foundry project ARM resource ID |
| `FOUNDRY_AGENT_NAME` | Name of the running agent |
| `FOUNDRY_AGENT_VERSION` | Version of the running agent |
| `FOUNDRY_AGENT_SESSION_ID` | Session ID for the current request |
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | App Insights connection string for telemetry |

### Reference Project Connections in Environment Variables

Pull secrets from Foundry project connections using placeholder syntax:

```yaml
environment_variables:
  - name: MODEL_DEPLOYMENT_NAME
    value: gpt-5-mini
  - name: GITHUB_TOKEN
    value: ${{connections.agent-secrets.credentials.github_token}}
```

| Placeholder Path | Resolves To |
|-----------------|-------------|
| `credentials.<field>` | A secret field on the connection |
| `target` | The connection's target property (e.g., endpoint URL) |
| `metadata.<field>` | A field under the connection's metadata |

> **Important:** Create the connection before deploying. If missing at sandbox start, the variable is empty.

## 1.3 Build and Push Your Container Image

```bash
# Build for linux/amd64 (required even on Apple Silicon)
docker build --platform linux/amd64 -t myagent:v1 .

# Login to Azure Container Registry
az acr login --name myregistry

# Tag and push
docker tag myagent:v1 myregistry.azurecr.io/myagent:v1
docker push myregistry.azurecr.io/myagent:v1
```

> **Tip:** Use unique image tags (`:v1`, `:v2`) instead of `:latest` for reproducible deployments.

### Configure Container Registry Permissions

1. In the Azure portal, go to your Foundry project resource
2. Select **Identity** and copy the **Object (principal) ID** under System assigned
3. Assign **Container Registry Repository Reader** role to this identity on your ACR

## 1.4 Deploy Using Python SDK

In [ ]:
# Deploy a Hosted Agent using Python SDK
print("""
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = "your_project_endpoint"

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

# Create a hosted agent version
agent = project.agents.create_version(
    agent_name="my-hosted-agent",
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1",
        memory="2Gi",
        image="your-registry.azurecr.io/your-image:v1",
        environment_variables={
            "MODEL_DEPLOYMENT_NAME": "gpt-5-mini"
        }
    )
)

print(f"Agent created: {agent.name}, version: {agent.version}")
""")

print("Key parameters:")
print("┌───────────────────────────────┬────────────────────────────────────────────────┐")
print("│ Parameter                     │ Description                                    │")
print("├───────────────────────────────┼────────────────────────────────────────────────┤")
print('│ agent_name                    │ Unique name (alphanumeric + hyphens, max 63)   │')
print('│ image                         │ Full ACR image URL with tag                    │')
print('│ cpu                           │ CPU allocation (e.g., "1")                     │')
print('│ memory                        │ Memory allocation (e.g., "2Gi")                │')
print('│ container_protocol_versions   │ Protocols: responses, invocations, or both     │')
print("└───────────────────────────────┴────────────────────────────────────────────────┘")

In [ ]:
# Poll for version status until active
print("""
import time

while True:
    version_info = project.agents.get_version(
        agent_name="my-hosted-agent",
        agent_version=agent.version
    )
    status = version_info["status"]
    print(f"Status: {status}")

    if status == "active":
        print("Agent is ready!")
        break
    elif status == "failed":
        print(f"Provisioning failed: {version_info['error']}")
        break

    time.sleep(5)
""")

print("Version status values:")
print("┌────────────┬────────────────────────────────────────────────────────┐")
print("│ Status     │ Description                                            │")
print("├────────────┼────────────────────────────────────────────────────────┤")
print("│ creating   │ Infrastructure provisioning in progress (2-5 min)      │")
print("│ active     │ Agent is ready to serve requests                       │")
print("│ failed     │ Provisioning failed — check error field                │")
print("│ deleting   │ Version is being cleaned up                            │")
print("│ deleted    │ Version has been fully removed                         │")
print("└────────────┴────────────────────────────────────────────────────────┘")

In [ ]:
# Invoke the Hosted Agent (Responses protocol)
print("""
# Create an OpenAI client bound to the agent endpoint
openai_client = project.get_openai_client(agent_name="my-hosted-agent")

response = openai_client.responses.create(
    input="Hello! What can you do?",
)

print(response.output_text)
""")

print("---")
print("For the Invocations protocol, call the endpoint directly:")
print("""
import requests

token = credential.get_token("https://ai.azure.com/.default").token
url = f"{PROJECT_ENDPOINT}/agents/my-hosted-agent/endpoint/protocols/invocations"

response = requests.post(url, headers={
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "Foundry-Features": "HostedAgents=V1Preview"
}, params={"api-version": "v1"}, json={
    "message": "Process this task"
})

print(response.json())
""")

## 1.5 Deploy Using REST API

```bash
# Set up variables
BASE_URL="https://{account}.services.ai.azure.com/api/projects/{project}"
API_VERSION="v1"
TOKEN=$(az account get-access-token --resource https://ai.azure.com --query accessToken -o tsv)

# Create an agent (also creates version 1 and triggers provisioning)
curl -X POST "$BASE_URL/agents?api-version=$API_VERSION" \
  -H "Authorization: Bearer $TOKEN" \
  -H "Content-Type: application/json" \
  -d '{
    "name": "my-agent",
    "definition": {
      "kind": "hosted",
      "image": "myacr.azurecr.io/my-agent:v1",
      "cpu": "1",
      "memory": "2Gi",
      "container_protocol_versions": [
        {"protocol": "responses", "version": "1.0.0"}
      ],
      "environment_variables": {
        "MODEL_DEPLOYMENT_NAME": "gpt-5-mini"
      }
    }
  }'

# Invoke (Responses protocol)
curl -X POST "$BASE_URL/agents/my-agent/endpoint/protocols/openai/responses?api-version=$API_VERSION" \
  -H "Authorization: Bearer $TOKEN" \
  -H "Content-Type: application/json" \
  -d '{
    "input": "Hello! What can you do?",
    "store": true
  }'

# Create a new version with updated image
curl -X POST "$BASE_URL/agents/my-agent/versions?api-version=$API_VERSION" \
  -H "Authorization: Bearer $TOKEN" \
  -H "Content-Type: application/json" \
  -d '{
    "definition": {
      "kind": "hosted",
      "image": "myacr.azurecr.io/my-agent:v2",
      "cpu": "1",
      "memory": "2Gi",
      "container_protocol_versions": [
        {"protocol": "responses", "version": "1.0.0"}
      ]
    }
  }'
```

## 1.6 Local Testing

Before deploying to Foundry, test your container locally:

**Responses protocol:**
```http
POST http://localhost:8088/responses
Content-Type: application/json

{
    "input": "Where is Seattle?",
    "stream": false
}
```

**Invocations protocol:**
```http
POST http://localhost:8088/invocations
Content-Type: application/json

{
    "message": "Hello!"
}
```

## 1.7 Troubleshooting Deployment

| Error Code | HTTP | Solution |
|-----------|------|----------|
| `image_pull_failed` | 400 | Verify image URI + project managed identity has **Container Registry Repository Reader** |
| `SubscriptionIsNotRegistered` | 400 | Register the subscription provider |
| `InvalidAcrPullCredentials` | 401 | Fix managed identity or registry RBAC |
| `UnauthorizedAcrPull` | 403 | Provide correct credentials or identity |
| `AcrImageNotFound` | 404 | Correct image name/tag or publish image |
| `RegistryNotFound` | 400/404 | Fix registry DNS or network reachability |

> **Important:** The ACR must be reachable over its public endpoint. Private endpoints are NOT supported for Hosted agents.

---
# PART 2: MANAGE HOSTED AGENTS
---

The platform manages the container lifecycle automatically. Compute is provisioned when a request arrives and deprovisioned after the idle timeout (15 minutes). There are no manual start/stop operations.

## 2.1 View Agents and Versions

In [ ]:
# List all agents in a project (Python SDK)
print("""
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# List all agents
for agent in project.agents.list():
    print(agent.name)

# Get agent details
agent = project.agents.get(agent_name="my-agent")
print(f"Name: {agent.name}")
print(f"Status: {agent.versions['latest']['status']}")

# Get a specific version
agent_version = project.agents.get_version(
    agent_name="my-agent", agent_version="1"
)
print(f"Version: {agent_version.version}")

# List all versions
for version in project.agents.list_versions(agent_name="my-agent"):
    print(f"Version: {version.version}, Status: {version['status']}")
""")

print("REST equivalent:")
print('  GET  $BASE_URL/agents                              # List all agents')
print('  GET  $BASE_URL/agents/{name}                       # Get agent details')
print('  GET  $BASE_URL/agents/{name}/versions/{ver}        # Get specific version')
print('  GET  $BASE_URL/agents/{name}/versions              # List all versions')
print()
print("azd equivalent:")
print('  azd ai agent show')

## 2.2 Configure Traffic Routing (Canary Deployments)

Route traffic between agent versions for canary deployments or gradual rollouts.

In [ ]:
# Configure traffic routing — Python SDK
print("""
from azure.ai.projects.models import (
    AgentEndpoint,
    AgentEndpointProtocol,
    FixedRatioVersionSelectionRule,
    VersionSelector,
)

# Pin 100% traffic to version 1
endpoint_config = AgentEndpoint(
    version_selector=VersionSelector(
        version_selection_rules=[
            FixedRatioVersionSelectionRule(
                agent_version="1", traffic_percentage=100
            ),
        ]
    ),
    protocols=[AgentEndpointProtocol.RESPONSES],
)

project.beta.agents.patch_agent_details(
    agent_name="my-agent",
    agent_endpoint=endpoint_config,
)
""")

print("--- Canary deployment (90/10 split) ---")
print("""
# Split traffic: 90% to v1, 10% to v2
endpoint_config = AgentEndpoint(
    version_selector=VersionSelector(
        version_selection_rules=[
            FixedRatioVersionSelectionRule(agent_version="1", traffic_percentage=90),
            FixedRatioVersionSelectionRule(agent_version="2", traffic_percentage=10),
        ]
    ),
    protocols=[AgentEndpointProtocol.RESPONSES],
)
""")

## 2.3 View Logs and Monitor

**REST API — Stream container logs:**
```bash
az rest --method GET \
    --url "${BASE_URL}/agents/${AGENT_NAME}/versions/${VERSION}/sessions/${SESSION_ID}:logstream?api-version=${API_VERSION}" \
    --resource "${RESOURCE}" \
    --headers "Foundry-Features=HostedAgents=V1Preview" "Accept=text/event-stream"
```

Log stream returns SSE with `timestamp`, `stream` (`stdout`/`stderr`/`status`), and `message` fields.

| Timeout | Value |
|---------|-------|
| Max connection duration | 30 minutes |
| Idle timeout | 2 minutes |

**azd — Real-time monitoring:**
```bash
azd ai agent monitor
```

> **Note:** Container log viewing is not currently supported through the Python SDK.

## 2.4 Retrieve Agent Identity for RBAC

In [ ]:
# Retrieve agent identity principal ID — Python SDK
print("""
agent = project.agents.get(agent_name="my-agent")
agent_identity = agent.instance_identity["principal_id"]
print(f"Agent identity principal ID: {agent_identity}")
""")

print("Assign RBAC roles to the agent identity:")
print("""
# Assign Foundry User on the project:
az role assignment create \\
    --assignee-object-id "$AGENT_IDENTITY" \\
    --assignee-principal-type ServicePrincipal \\
    --role "Azure AI Developer" \\
    --scope "/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<acct>/projects/<proj>"

# Assign Storage Blob Data Contributor:
az role assignment create \\
    --assignee-object-id "$AGENT_IDENTITY" \\
    --assignee-principal-type ServicePrincipal \\
    --role "Storage Blob Data Contributor" \\
    --scope "/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<account>"
""")

## 2.5 Delete Agents

In [ ]:
# Delete operations
print("""
# Delete a specific version
project.agents.delete_version(agent_name="my-agent", agent_version="1")

# Delete the entire agent and ALL versions (irreversible!)
project.agents.delete(agent_name="my-agent")
""")

print("REST equivalents:")
print('  DELETE $BASE_URL/agents/{name}/versions/{ver}   # Delete specific version')
print('  DELETE $BASE_URL/agents/{name}                  # Delete agent + all versions')
print()
print("azd equivalent:")
print('  azd down    # Full cleanup of all provisioned resources')

---
# PART 3: MANAGE HOSTED SESSIONS
---

## 3.1 Sessions vs. Conversations

```
┌────────────────────────────────────────────────────────────────────────┐
│                    SESSIONS vs. CONVERSATIONS                         │
├────────────────────────────┬───────────────────────────────────────────┤
│         SESSION            │           CONVERSATION                   │
├────────────────────────────┼───────────────────────────────────────────┤
│ Sandbox compute +          │ History of messages, tool calls,         │
│ persisted filesystem       │ and responses                           │
│ ($HOME, /files)            │                                         │
├────────────────────────────┼───────────────────────────────────────────┤
│ Identifier:                │ Identifier:                             │
│ agent_session_id           │ previous_response_id or conversation    │
├────────────────────────────┼───────────────────────────────────────────┤
│ Used for: file uploads,    │ Used for: threading turns of a          │
│ working state across turns │ chat together                           │
├────────────────────────────┼───────────────────────────────────────────┤
│ Managed by: platform       │ Managed by: platform (Responses)        │
│ via /sessions API          │ or your container code (Invocations)    │
├────────────────────────────┼───────────────────────────────────────────┤
│ Persists: up to 30 days    │                                         │
│ Idle timeout: 15 minutes   │                                         │
└────────────────────────────┴───────────────────────────────────────────┘
```

### Key Distinction

- **Session continuity ≠ conversation continuity**
- For **Responses**: preserve message history via `previous_response_id` or `conversation` ID
- For **Invocations**: your container manages state; sessions only provide sandbox persistence
- `agent_session_id` ties calls to the same sandbox, not to the same chat history

### How Each Protocol Binds to a Session

| Protocol | Endpoint | Where to put `agent_session_id` |
|----------|---------|-------------------------------|
| **Responses** | `POST .../protocols/openai/responses` | Request body field `agent_session_id` (or use `conversation` which autobinds) |
| **Invocations** | `POST .../protocols/invocations` | Query string: `?agent_session_id=<id>` |

## 3.2 Invoke and Let Platform Create Session

In [ ]:
# Invoke with auto-created session — Responses protocol (Python SDK)
print("""
openai_client = project.get_openai_client(agent_name="my-agent")

# First call — platform creates a session automatically
response = openai_client.responses.create(
    input="Find me hotels in Seattle under $200 per night",
)
session_id = response.model_extra.get("agent_session_id")
print(f"Session: {session_id}")
print(f"Response: {response.output_text}")

# Reuse the session + thread conversation on a later turn
follow_up = openai_client.responses.create(
    input="Recommend one of those hotels",
    previous_response_id=response.id,
    extra_body={"agent_session_id": session_id},
)
print(follow_up.output_text)
""")

print("--- Alternative: Using conversation ID (auto-binds session) ---")
print("""
conversation = openai_client.conversations.create()

first = openai_client.responses.create(
    input="Find me hotels in Seattle under $200 per night",
    extra_body={"conversation": conversation.id},
)
follow_up = openai_client.responses.create(
    input="Recommend one of those hotels",
    extra_body={"conversation": conversation.id},
)
# No need to pass agent_session_id — platform auto-binds!
""")

## 3.3 Isolation Keys

The isolation key scopes which sessions a caller can see and operate on.

| Scheme | How Key Is Set | Details |
|--------|---------------|--------|
| **Entra** (default) | Auto-derived from caller's Entra token | Each authenticated caller gets their own scope automatically |
| **Header** | App sends `x-ms-user-isolation-key` header | Stable string per user/tenant on every request; platform doesn't validate |

> **Important:** Isolation keys are partitioning, NOT authentication. The Entra token authenticates; the key narrows scope.

## 3.4 Create a Session Explicitly (Advanced)

Only needed when you must:
- Upload files before the agent's first turn
- Preallocate a session for your client code
- Pin the session to a specific agent version

In [ ]:
# Create session explicitly — Python SDK
print("""
# Basic session creation
session = project.beta.agents.create_session(
    agent_name="my-agent",
    body={},
    isolation_key="user-123",
)
print(f"Session created (ID: {session.agent_session_id}, status: {session.status})")

# Pin session to a specific agent version
session = project.beta.agents.create_session(
    agent_name="my-agent",
    body={
        "version_indicator": {"type": "version_ref", "agent_version": "2"},
    },
    isolation_key="user-123",
)

# List sessions
sessions = project.beta.agents.list_sessions(agent_name="my-agent")
for item in sessions:
    print(f"Session: {item.agent_session_id} (status: {item.status})")

# Get session details
session = project.beta.agents.get_session(
    agent_name="my-agent",
    session_id="<session-id>",
)

# Delete a session
project.beta.agents.delete_session(
    agent_name="my-agent",
    session_id="<session-id>",
    isolation_key="user-123",
)
""")

## 3.5 Session File Operations

Upload and download files to agent session sandboxes. Each file is scoped to a specific session. **Max file size: 50 MB.**

> **Note:** The Python SDK uses `session_id` for `upload_session_file` and `agent_session_id` for `get_session_files`, `download_session_file`, and `delete_session_file`.

In [ ]:
# Session file operations — Python SDK
print("""
# Upload a file
project.beta.agents.upload_session_file(
    agent_name="my-agent",
    session_id="<session-id>",
    content_or_file_path="./data.csv",
    path="data.csv",
)

# List files in a session
files = project.beta.agents.get_session_files(
    agent_name="my-agent",
    agent_session_id="<session-id>",
    path=".",
)
for entry in files.entries:
    print(f"  {entry['name']} (size: {entry['size']}, directory: {entry['is_directory']})")

# Download a file
content_bytes = b"".join(
    project.beta.agents.download_session_file(
        agent_name="my-agent",
        agent_session_id="<session-id>",
        path="data.csv",
    )
)
with open("./output.csv", "wb") as f:
    f.write(content_bytes)

# Delete a file
project.beta.agents.delete_session_file(
    agent_name="my-agent",
    agent_session_id="<session-id>",
    path="data.csv",
)
""")

print("azd equivalents:")
print('  azd ai agent files upload --file ./data.csv --target-path data.csv')
print('  azd ai agent files list .')
print('  azd ai agent files download --file data.csv --target-path ./output.csv')
print('  azd ai agent files remove --file data.csv')

---
# PART 4: PUBLISH TO MICROSOFT 365 COPILOT & TEAMS
---

## 4.1 Publishing Overview

Publishing lets users discover and interact with your agent through Microsoft 365 Copilot and Teams. What gets published is the agent's **stable endpoint** — you can roll out new versions without republishing.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    PUBLISHING FLOW                                   │
│                                                                      │
│  ┌─────────────┐    ┌──────────────────┐    ┌────────────────────┐  │
│  │ 1. SELECT    │    │ 2. CONFIGURE     │    │ 3. PUBLISH         │  │
│  │    ACTIVE    │───▶│    METADATA       │───▶│                    │  │
│  │    VERSION   │    │                  │    │ Direct publish     │  │
│  │              │    │ • Name           │    │ OR                 │  │
│  │ Always latest│    │ • Description    │    │ Download & sideload│  │
│  │ or pinned    │    │ • Developer info │    │                    │  │
│  └─────────────┘    └──────────────────┘    └────────────────────┘  │
│                                                       │              │
│                                              ┌────────▼────────┐     │
│                                              │  VISIBILITY     │     │
│                                              │                 │     │
│                                              │ Just you        │     │
│                                              │ (immediate)     │     │
│                                              │                 │     │
│                                              │ Organization    │     │
│                                              │ (admin approval)│     │
│                                              └─────────────────┘     │
└──────────────────────────────────────────────────────────────────────┘
```

### Prerequisites for Publishing

| Requirement | Details |
|------------|--------|
| **Foundry User** role | On the Foundry project scope |
| **Azure Bot Service** | `Microsoft.BotService` provider registered in subscription |
| **Agent tested** | Thoroughly tested in Foundry portal before publishing |
| **Active version** | Selected which version should receive traffic |

## 4.2 Publishing Steps

### Step 1: Select Active Version

1. In the Foundry portal, select **Publish**
2. Click the arrow next to **Active version**
3. Choose "Always use latest" or select a specific version

### Step 2: Configure Metadata

| Field | Description |
|-------|------------|
| **Name** | Display name in agent store |
| **Publish version** | Three-part version (major.minor.patch) |
| **Short description** | One sentence about what agent does |
| **Description** | Longer description of responsibilities and actions |
| **Developer** | Your name or organization |

Optional metadata:

| Field | Description |
|-------|------------|
| **Developer website** | URL (HTTPS required) |
| **Terms of use** | URL (HTTPS required) |
| **Privacy statement** | URL (HTTPS required) |

> **Warning:** Don't include secrets, API keys, or sensitive information in metadata fields.

### Step 3: Choose Publishing Method

#### Option A: Direct Publish

| Scope | Behavior | Admin Approval | Best For |
|-------|---------|---------------|----------|
| **Just you** | Available immediately. Appears under "Your agents" in agent store | Not required | Personal testing, small teams, pilots |
| **People in your organization** | Submitted for admin approval. Appears under "Built by your org" once approved | Required | Organization-wide distribution, production |

#### Option B: Download & Customize

1. Download the `.zip` manifest file
2. Customize the manifest as needed
3. In Microsoft Teams: **Apps** > **Manage your apps** > **Upload an app** > **Upload a custom app**

## 4.3 Update a Published Agent

### Update the Active Version
Change the version selector in Foundry portal. The stable endpoint URL stays the same — **no need to republish** to M365/Teams.

### Update Metadata
In the **Publish** dropdown, select **Update agent Teams and Microsoft 365 Copilot display properties**. The version auto-increments if not manually changed.

### Publishing Limitations

| Limitation | Description |
|-----------|------------|
| File uploads & image generation in M365 | Not supported (works in Teams only) |
| Private Link | Not supported for Teams or Bot Service integrations |
| Streaming & citations | Not supported for published agents |

## 4.4 Troubleshooting Publishing

| Issue | Cause | Resolution |
|-------|-------|------------|
| Error publishing the agent | Invalid metadata | Ensure agent has unique identity; developer name ≤ 32 chars |
| Azure Bot Service creation fails | Missing permissions | Register `Microsoft.BotService` provider. Assign **Azure Bot Service Contributor** on resource group |
| 403 AuthorizationFailed for BotService | Missing role | Assign **Azure Bot Service Contributor** role on the resource group |
| Organization scope agent doesn't appear | Admin approval pending | Check M365 admin center for approval |
| Agent works in Foundry but fails after publishing | Missing RBAC | Assign roles to the agent's identity for accessed resources |
| Publishing fails with identity error | `agent.identity` is null | Follow migration guide |
| Users can't find the agent | Wrong scope or pending approval | For Individual, share direct link. For Org, confirm admin approval |

---
# PART 5: MICROSOFT AGENT 365 (A365)
---

## 5.1 What Is Agent 365?

**Microsoft Agent 365 (A365)** is Microsoft's IT admin control plane for AI agents. It provides identity, security, governance, and lifecycle management for agents at scale.

### Core Capabilities

| Capability | Description |
|-----------|------------|
| **Registry** | Complete inventory of agents (Foundry, Copilot Studio, SaaS agents, shadow agents) |
| **Access control** | Entra-based controls, Conditional Access policies, network controls |
| **Visualization** | Explore connections between agents, people, and data; monitor behavior |
| **Interoperability** | Connect agents to M365 apps and organizational data |
| **Security** | Protect against threats, data oversharing, leaks, and risky behavior |

### Foundry + Agent 365 Integration

**All Foundry agents automatically appear in the Agent 365 agent registry on creation.** No extra configuration required. Admins see:
- Agent name, description, tools
- Agent identity and blueprint
- Unified view alongside Copilot Studio and other ecosystem agents

## 5.2 AI Teammates

Foundry Hosted agents can be pushed as **AI teammates** to Agent 365. Once approved by an admin, they can be "hired" by others in the organization.

### Prerequisites for AI Teammates

| Requirement | Details |
|------------|--------|
| **Microsoft 365 E7** | Required license |
| **Owner** role | On the Azure subscription |
| **Foundry User** | At subscription or resource group scope |
| **Tenant admin** | Can approve agent requests in M365 admin center |
| **Hosted agents region** | Must use a supported region |
| **Docker, .NET 9.0 SDK** | For building the container |

### What the Sample Creates

1. Azure resources required to run the agent
2. Agent version with endpoint traffic routing
3. AI teammate request requiring admin approval

### Run the Code Sample

```bash
# Clone the foundry-samples repository
git clone https://github.com/microsoft-foundry/foundry-samples.git
cd samples/csharp/FoundryA365

# Follow the azd workflow
az login
azd auth login
azd provision
azd env get-values
```

### Validate

1. **Approve** the blueprint request in M365 admin center: `https://admin.cloud.microsoft/?#/agents/all/requested`
2. **Verify** the agent appears in Agent 365 registry
3. **Configure** Teams integration in the Teams Developer Portal
4. In Teams: **Apps** > **Agents for your team** > Find and create an instance

---
# PART 6: AGENT-TO-AGENT (A2A) PROTOCOL
---

## 6.1 What Is A2A?

The **Agent2Agent (A2A)** protocol lets agents discover and call each other. When enabled on a Foundry agent, the platform publishes an **agent card** and accepts inbound A2A requests.

```
┌───────────────────────────────────────────────────────────────────┐
│                    A2A COMMUNICATION FLOW                        │
│                                                                   │
│  ┌──────────────┐         ┌──────────────────────┐               │
│  │ CALLING       │  A2A   │ TARGET AGENT          │               │
│  │ AGENT         │───────▶│                      │               │
│  │               │        │ Agent Card            │               │
│  │ Uses          │        │ ├── Description       │               │
│  │ A2APreviewTool│        │ ├── Version           │               │
│  │               │        │ └── Skills            │               │
│  │               │◀───────│                      │               │
│  │               │ result │ Protocols:            │               │
│  └──────────────┘         │ ├── responses         │               │
│                           │ └── a2a               │               │
│                           └──────────────────────┘               │
│                                                                   │
│  Authentication: Microsoft Entra ID (required)                    │
│  Transports: HTTP+JSON ✔️  JSONRPC ✔️  gRPC ❌                    │
│  Protocol version: 0.3 only                                       │
│  Modality: Text only (no files)                                   │
└───────────────────────────────────────────────────────────────────┘
```

### Supported Agent Types for A2A

| Agent Type | A2A Support |
|-----------|------------|
| **Prompt agents** | Supported (use responses protocol by default) |
| **Hosted agents** | Only if built to handle responses protocol |

## 6.2 Enable Incoming A2A

Two things are needed: an **agent card** and the **A2A protocol** enabled on the endpoint.

In [ ]:
# Enable A2A — Python SDK (endpoint config only; agent card requires REST)
print("""
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AgentEndpoint,
    AgentEndpointProtocol,
)

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

endpoint_config = AgentEndpoint(
    protocols=[
        AgentEndpointProtocol.RESPONSES,
        AgentEndpointProtocol.A2A,
    ],
)

patched_agent = project_client.beta.agents.patch_agent_details(
    agent_name="my-agent",
    agent_endpoint=endpoint_config,
)
print(f"A2A enabled for: {patched_agent.name}")
""")

print("--- Configure agent card via REST API ---")
print("""
curl -X PATCH "$BASE_URL/agents/$AGENT_NAME?api-version=v1" \\
  -H "Authorization: Bearer $TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{
    "agent_card": {
      "description": "A helpful assistant that answers questions",
      "version": "1.0",
      "skills": [
        {
          "id": "general-qa",
          "name": "General Q&A",
          "description": "Answers general questions"
        }
      ]
    },
    "agent_endpoint": {
      "protocols": ["responses", "a2a"]
    }
  }'
""")

## 6.3 A2A Endpoint URLs

After enabling A2A, your agent exposes:

| URL | Purpose |
|-----|--------|
| **A2A base path** | `https://{account}.services.ai.azure.com/api/projects/{project}/agents/{agent}/endpoint/protocols/a2a` |
| **Agent card URL** | `https://{account}.services.ai.azure.com/api/projects/{project}/agents/{agent}/endpoint/protocols/a2a/agentCard/v0.3` |

> **Important:** Both URLs require Microsoft Entra ID authentication. Anonymous access is NOT supported. The calling agent needs **Foundry User** role on the project.

### Verify Agent Card

```bash
curl -X GET "$BASE_URL/agents/$AGENT_NAME/endpoint/protocols/a2a/agentCard/v0.3" \
  -H "Authorization: Bearer $TOKEN"
```

## 6.4 Authentication for A2A

| Pattern | Description | Use Case |
|---------|-----------|----------|
| **On-behalf-of (OBO)** | Calling agent passes through end user's identity | Per-user access control |
| **Service identity** | Calling agent uses its own identity (agent identity, service principal, or managed identity) | Backend agent-to-agent workflows |

## 6.5 Connect FROM Another Foundry Agent

### Step 1: Create an A2A Connection

```bash
# Create A2A connection via REST (required for custom agent card path)
curl --request PUT \
  --url "https://management.azure.com/subscriptions/$SUB_ID/resourceGroups/$RG/providers/Microsoft.CognitiveServices/accounts/$ACCOUNT/projects/$PROJECT/connections/$CONNECTION_NAME?api-version=2025-04-01-preview" \
  --header "Authorization: Bearer $TOKEN" \
  --header "Content-Type: application/json" \
  --data '{
    "properties": {
      "authType": "AgenticIdentity",
      "category": "RemoteA2A",
      "target": "https://{account}.services.ai.azure.com/api/projects/{project}/agents/{agent}/endpoint/protocols/a2a",
      "audience": "https://ai.azure.com",
      "Credentials": {},
      "metadata": {
        "AgentCardPath": "/agentCard/v0.3"
      }
    }
  }'
```

> **Key:** The `AgentCardPath` metadata is required because Foundry uses `/agentCard/v0.3` instead of the default `.well-known/agent-card.json`.

### Step 2: Create the Calling Agent with A2APreviewTool

In [ ]:
# Create calling agent with A2A tool — Python SDK
print("""
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    A2APreviewTool,
)

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai = project.get_openai_client()

# Get the A2A connection
a2a_connection = project.connections.get("my-a2a-target")

# Create agent with A2A tool
tool = A2APreviewTool(
    project_connection_id=a2a_connection.id,
)

agent = project.agents.create_version(
    agent_name="my-calling-agent",
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",
        instructions=(
            "You are a helpful assistant. Use the A2A tool "
            "to delegate tasks to the target agent."
        ),
        tools=[tool],
    ),
)

# Send a message and stream the response
stream_response = openai.responses.create(
    stream=True,
    input="Ask the target agent what it can do.",
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    },
)

for event in stream_response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.completed":
        print(f"\\n\\nCompleted: {event.response.output_text}")

# Clean up
project.agents.delete_version(
    agent_name=agent.name, agent_version=agent.version
)
""")

## 6.6 Connect Using the Python A2A SDK

Use the open-source [Python A2A SDK](https://github.com/a2aproject/a2a-python) with Entra authentication:

```bash
pip install a2a-sdk==1.0.2 azure-identity==1.25.3 httpx==0.28.1
```

In [ ]:
# Connect to Foundry A2A agent with Python A2A SDK
print("""
import asyncio
import httpx
from azure.identity import DefaultAzureCredential
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import new_text_message
from a2a.types.a2a_pb2 import Role, SendMessageRequest

A2A_BASE_URL = (
    "https://{account}.services.ai.azure.com/api/projects"
    "/{project}/agents/{agent}/endpoint/protocols/a2a"
)
AGENT_CARD_PATH = "agentCard/v0.3"

async def main():
    credential = DefaultAzureCredential()
    token = credential.get_token("https://ai.azure.com/.default").token

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(120.0),
    ) as httpx_client:
        # Resolve the agent card
        resolver = A2ACardResolver(
            httpx_client=httpx_client,
            base_url=A2A_BASE_URL,
            agent_card_path=AGENT_CARD_PATH,
        )
        agent_card = await resolver.get_agent_card()

        # Create a non-streaming A2A client
        config = ClientConfig(streaming=False, httpx_client=httpx_client)
        client = await create_client(agent=agent_card, client_config=config)

        # Send a message
        message = new_text_message("Hello, what can you do?", role=Role.ROLE_USER)
        request = SendMessageRequest(message=message)

        async for response in client.send_message(request):
            print(response)

        await client.close()

asyncio.run(main())
""")

## 6.7 A2A Limitations

| Limitation | Details |
|-----------|--------|
| Protocol version | Only **v0.3** supported |
| Modality | **Text only** — no files or non-text modalities |
| Agent requirement | Must use responses protocol |
| gRPC transport | Not supported |
| Status | Preview — not recommended for production workloads |

---
# PART 7: INTERVIEW Q&A
---

### Q1: What is the deployment lifecycle for a Hosted agent?
**A:** (1) **Build & push** — package code into a container image and push to Azure Container Registry. (2) **Create agent version** — register the image with Foundry Agent Service; platform provisions infrastructure and creates a dedicated Entra agent identity. (3) **Poll for status** — wait until version status is `active`. (4) **Invoke** — send requests to the agent's dedicated endpoint via Responses or Invocations protocol.

---

### Q2: What are the two protocols for Hosted agents and when would you use each?
**A:** **Responses** — OpenAI-compatible, platform manages conversation history, streaming, and background execution. Best for chatbots, Q&A, RAG, multi-turn conversations. **Invocations** — accepts arbitrary JSON payloads, gives full HTTP/SSE control. Best for webhook receivers (GitHub, Stripe), non-conversational processing, custom async workflows. A single container can expose both simultaneously.

---

### Q3: What is the difference between a session and a conversation in Hosted agents?
**A:** A **session** is a stateful, isolated sandbox with persisted filesystem (`$HOME` and `/files`). It persists up to 30 days with a 15-minute idle timeout. A **conversation** is the history of messages, tool calls, and responses. Session continuity ≠ conversation continuity. For Responses protocol, preserve chat history via `previous_response_id` or `conversation` ID. For Invocations, your container manages state. `agent_session_id` only ties calls to the same sandbox, not the same chat.

---

### Q4: How do isolation keys work for Hosted agent sessions?
**A:** Isolation keys scope which sessions a caller can see and operate on. Two modes: (1) **Entra** (default) — platform auto-derives the key from the caller's Entra token; each authenticated caller gets their own scope. (2) **Header** — app sends `x-ms-user-isolation-key` header with a stable string per user/tenant. The key is a partitioning value, NOT authentication — Entra tokens handle auth.

---

### Q5: How do you configure canary deployments for Hosted agents?
**A:** Use traffic routing with `FixedRatioVersionSelectionRule`. Create multiple agent versions, then patch the agent endpoint with version selection rules specifying traffic percentages (e.g., 90% to v1, 10% to v2). The platform routes incoming requests accordingly. Use the Python SDK's `patch_agent_details` or REST API's `PATCH /agents/{name}` with `agent_endpoint.version_selector.version_selection_rules`.

---

### Q6: What happens to agent identity when deploying a Hosted agent?
**A:** The platform creates a **dedicated Entra agent identity** (service principal) for each Hosted agent at deploy time. This identity is used by the running container to call models and tools. The deploying user must have **Foundry Project Manager** role because they need permission to assign **Foundry User** to the platform-created identity. The identity's principal ID can be retrieved via `agent.instance_identity["principal_id"]` and used for RBAC assignments.

---

### Q7: What are the steps to publish a Foundry agent to Microsoft 365 Copilot and Teams?
**A:** (1) Test thoroughly in Foundry portal. (2) Select active agent version ("Always latest" or pinned). (3) Register `Microsoft.BotService` provider. (4) Click **Publish** > **Publish to Teams and Microsoft 365 Copilot**. (5) Fill in metadata (name, description, developer). (6) Choose scope: "Just you" (immediate) or "Organization" (requires admin approval in M365 admin center). (7) Alternatively, download manifest ZIP and sideload in Teams.

---

### Q8: What is Microsoft Agent 365 and how does it integrate with Foundry?
**A:** Agent 365 (A365) is Microsoft's IT admin control plane for AI agents. It provides: agent registry (inventory), access control (Entra + Conditional Access), visualization (agent-people-data connections), interoperability (M365 apps), and security. **All Foundry agents automatically appear in the A365 registry on creation** — no configuration needed. Hosted agents can also be pushed as **AI teammates** that users can "hire" in Teams after admin approval.

---

### Q9: How does Agent-to-Agent (A2A) work in Foundry?
**A:** A2A lets agents discover and call each other via the A2A protocol (v0.3). To expose an agent: (1) Configure an **agent card** with description and skills. (2) Enable the A2A protocol on the endpoint alongside responses. (3) The platform publishes discovery URLs. To call a remote A2A agent: (1) Create an **A2A connection** with the target's endpoint URL, auth type, and custom `AgentCardPath`. (2) Create a calling agent with `A2APreviewTool` referencing the connection. Authentication is always Entra ID — supports OBO (user passthrough) or service identity.

---

### Q10: What environment variables does the Hosted agent platform inject automatically?
**A:** The platform injects: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_PROJECT_ARM_ID`, `FOUNDRY_AGENT_NAME`, `FOUNDRY_AGENT_VERSION`, `FOUNDRY_AGENT_SESSION_ID`, and `APPLICATIONINSIGHTS_CONNECTION_STRING`. The `FOUNDRY_*` prefix is reserved for platform use. Don't redeclare these in `agent.yaml`. Custom variables (like `MODEL_DEPLOYMENT_NAME`) go in the `environment_variables` section. Secrets can be pulled from project connections using `${{connections.<name>.credentials.<field>}}` placeholder syntax.

---

### Q11: How do you reference secrets in Hosted agent environment variables without hardcoding?
**A:** Use **placeholder expressions** in `agent.yaml`: `${{connections.<name>.<path>}}`. The platform resolves these at sandbox start. Supported paths: `credentials.<field>` (secret), `target` (endpoint URL), `metadata.<field>`. For `ApiKey` connections, use `credentials.key`. For `CustomKeys`, use the field name you supplied at creation. The resolved secret is never echoed back via GET — secrets are write-only. Create the connection before deploying; missing connections result in empty variables.

---

### Q12: What are the key differences between Responses and Invocations protocols for session binding?
**A:** For **Responses**: `agent_session_id` goes in the request body field. Alternatively, use a `conversation` ID which autobinds a session. The platform manages conversation history. For **Invocations**: `agent_session_id` goes in the **query string** only — body fields and headers named `session_id` are passed through to the container but don't affect routing. The platform doesn't store conversation history; your container manages state. In both cases, omitting the session ID creates a new session.

---

### Q13: What are the limitations of publishing agents to M365/Teams?
**A:** (1) File uploads and image generation don't work in M365 (work in Teams only). (2) Private Link is not supported for Teams or Bot Service integrations. (3) Streaming responses and citations are not supported for published agents. (4) Agent cards must have unique identities. (5) Organization-scope agents require M365 admin approval.

---

### Q14: What container image requirements must Hosted agents meet?
**A:** (1) Must be `linux/amd64` architecture (use `--platform linux/amd64` on ARM). (2) Must implement at least one protocol library (`azure-ai-agentserver-responses` or `azure-ai-agentserver-invocations`). (3) Serves traffic on port **8088** locally. (4) Protocol libraries auto-expose `/readiness` for health checks. (5) ACR must be reachable over public endpoint (private endpoints not supported). (6) Use unique image tags, not `:latest`.

---

### Q15: How do you use the A2A protocol from a non-Foundry client (e.g., Python A2A SDK)?
**A:** Install `a2a-sdk`, `azure-identity`, and `httpx`. Configure an `httpx.AsyncClient` with a Bearer token from `DefaultAzureCredential`. Create an `A2ACardResolver` with the A2A base URL and custom `agent_card_path="agentCard/v0.3"` (Foundry uses a non-standard path). Resolve the agent card, create a client with `create_client()`, then call `send_message()` with a text message. The SDK auto-uses v0.3 compatibility mode based on the agent card's protocol version.

---
## Clean Up Resources

In [ ]:
# Clean up
print("""
# Python SDK
project.agents.delete_version(agent_name="my-hosted-agent", agent_version="1")
# OR delete entire agent:
project.agents.delete(agent_name="my-hosted-agent")

# REST API
# curl -X DELETE "$BASE_URL/agents/my-agent/versions/1?api-version=$API_VERSION" -H "Authorization: Bearer $TOKEN"
# curl -X DELETE "$BASE_URL/agents/my-agent?api-version=$API_VERSION" -H "Authorization: Bearer $TOKEN"

# azd
# azd down
""")

print("Note: Agent compute is deprovisioned after 15 minutes of inactivity.")
print("No cost is incurred when an agent isn't serving requests.")

---
## Summary

In this notebook you learned:

- [x] Hosted agent deployment lifecycle: build, push, create version, poll, invoke
- [x] Container requirements: protocols (Responses/Invocations), ports, platform-injected env vars
- [x] Secret management via project connection placeholders
- [x] Deploy using Python SDK, REST API, and Azure Developer CLI
- [x] Manage agents: list, version, create new versions, delete
- [x] Traffic routing and canary deployments with version selection rules
- [x] Container log streaming and monitoring
- [x] Agent identity retrieval and RBAC role assignments
- [x] Sessions vs. conversations: distinct concepts for sandbox vs. chat history
- [x] Session management: create, list, get, delete, file operations
- [x] Isolation keys: Entra (auto) vs. Header (app-supplied) scoping
- [x] Publish to Microsoft 365 Copilot & Teams: direct publish and sideload
- [x] Publishing scopes: Just you (immediate) vs. Organization (admin approval)
- [x] Microsoft Agent 365: IT admin governance, registry, AI teammates
- [x] Agent-to-Agent (A2A): expose agents as A2A endpoints with agent cards
- [x] A2A connections: create connections and calling agents with A2APreviewTool
- [x] A2A authentication: OBO and service identity patterns
- [x] 15 interview-ready Q&A covering all core concepts

### Useful Links

- [Deploy a Hosted Agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent)
- [Manage Hosted Agents](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent)
- [Manage Hosted Sessions](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions)
- [Publish to M365 Copilot & Teams](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/publish-copilot)
- [Microsoft Agent 365](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/agent-365)
- [Enable A2A Endpoint](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint)
- [Hosted Agent Samples](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/hosted-agents)
- [A2A Protocol](https://a2a-protocol.org/latest/)
- [Python A2A SDK](https://github.com/a2aproject/a2a-python)
- [Foundry Portal](https://ai.azure.com)